# Full pipeline: residual key → panel → simulation

One notebook, start to finish: a `residual_key` string and a `candidate_panel_path` in, a `SimulationResult` out. It downloads market data if it isn't cached yet, builds and persists the candidate panel, then simulates against it. For the panel-building step in detail, see [01_create_candidate_panel.ipynb](01_create_candidate_panel.ipynb); for result inspection, see [02_run_simulation.ipynb](02_run_simulation.ipynb).

In [ ]:
# ── knobs ─────────────────────────────────────────────────────────────
RESIDUAL_KEY = "exp_hl504_mh1008_rf"   # parsed via CausalResidualConfig.from_key
CANDIDATE_PANEL_PATH = "howto3"        # subdir under CANDIDATE_PANELS_ROOT — panel build and
                                        # simulation both read/write here; defined once, reused below
SELECTED_SECTORS = ["materials"]
ENTRY_Z = 1.5
Z_LOOKBACK = 21
Z_METHOD = "ewm"

# panel-build windows — PanelBatchConfig's concern, independent of the z-score above
HEDGE_RATIO_LB = 252
MR_DIAG_LB = 252
MAX_STEPS = 60                         # notebook-speed cap on panel creation (like notebook 01)


## 1. Ensure market data is present (download only if missing)

In [ ]:
from src.data.universe_config import UniverseConfig
from src.data.universe_loader import UniverseDataLoader
from src.settings import CONFIG_UNIVERSE, DATA_UNIVERSES

for sector in SELECTED_SECTORS:
    yaml_path = CONFIG_UNIVERSE / f"universe.{sector}_only.v1.yaml"
    ucfg = UniverseConfig.from_yaml(yaml_path)
    prices_path = DATA_UNIVERSES / ucfg.universe_name / "prices_daily.parquet"
    if prices_path.exists():
        print(f"[download] {sector}: cached at {prices_path}")
    else:
        print(f"[download] {sector}: not found — downloading...")
        UniverseDataLoader(ucfg, data_path=DATA_UNIVERSES, progress=True).load(force_download=False)


## 2. Build and persist the candidate panel

In [ ]:
from src.candidates.panel_batch import PanelBatchConfig, run_panel_batch

panel_cfg = PanelBatchConfig(
    residual_configs=[RESIDUAL_KEY],   # list[str] — resolved via CausalResidualConfig.from_key
    hedge_ratio_lb=HEDGE_RATIO_LB,
    mr_diag_lb=MR_DIAG_LB,
    selected_sectors=SELECTED_SECTORS,
    frequency="W-FRI",
    max_steps=MAX_STEPS,
    persist_result=True,
    persist_residual_params=True,
    persist_dir_template=CANDIDATE_PANEL_PATH,
)
panel_results = run_panel_batch(panel_cfg)
print(f"panels built: {list(panel_results.keys())}")


## 3. Run the simulation against the panel just built

`SweepConfig` + `run_sweep` is the minimal simulator entry point. Even with a single
residual timescale, `RESIDUAL_KEY` must be threaded through `z_score_overrides` (not the
plain `z_lookback`/`z_method` fields) — the panel just built carries its real,
non-empty `residual_key`, and the simulator matches `ZScoreConfig.residual_key` against
it to select the panel. `start_date`/`end_date` are left unset so the run covers exactly
the (small, capped) date range the panel above was built over.

In [ ]:
from src.simulator.config import ZScoreConfig
from src.simulator.sweep_runner import SweepConfig, run_sweep

RUNS = [
    SweepConfig(
        entry_z=ENTRY_Z,
        z_score_overrides=[ZScoreConfig(lookback=Z_LOOKBACK, method=Z_METHOD, residual_key=RESIDUAL_KEY)],
        candidate_panel_subdir=CANDIDATE_PANEL_PATH,
        start_date=None,
        end_date=None,
    ),
]
# skip_existing=False: this howto should always produce a fresh result to inspect below,
# even if an identical config is already sitting in sweep_results.pkl from a prior run.
sweep_df, sweep_results = run_sweep(RUNS, skip_existing=False)
result = sweep_results[0]


## 4. Result summary

In [ ]:
print(f"closed trades: {len(result.closed_trades)}")

m = result.performance.metrics
for k in ["n_trades", "win_rate_net", "sharpe_net", "total_net_pnl", "max_drawdown_net"]:
    if k in m:
        print(f"  {k:18s}: {m[k]}")
